# CPI Prediction — Data Collection

Collects data from FRED, Philadelphia Fed SPF, Kalshi, GDELT, and Google Trends.
Outputs `data/processed/monthly_panel.csv` used by `02_analysis.ipynb`.

**API keys needed:** FRED (`FRED_API_KEY`) and Kalshi (`KALSHI_EMAIL`, `KALSHI_PASSWORD`). Add them to `.env` before running those cells.

In [ ]:
import os, io, re, time, json, warnings
import requests
import numpy as np
import pandas as pd
from io import StringIO

warnings.filterwarnings('ignore')

# Run from project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
print("Working directory:", os.getcwd())

## 1. FRED Macro & Survey Data

Needs a free API key from https://fredaccount.stlouisfed.org/apikey

Set `FRED_API_KEY` in a `.env` file in the project root, or paste it directly below.

In [ ]:
# ── SET YOUR FRED API KEY HERE ──────────────────────────────────
FRED_API_KEY = os.getenv('FRED_API_KEY', 'YOUR_FRED_API_KEY_HERE')
# ────────────────────────────────────────────────────────────────

FRED_BASE = 'https://api.stlouisfed.org/fred/series/observations'
START, END = '2021-01-01', '2026-04-30'

FRED_SERIES = {
    'cpi_level':           'CPIAUCSL',
    'michigan_inflation':  'MICH',
    'cleveland_inflation': 'EXPINF1YR',
    'unemployment':        'UNRATE',
    'ppi':                 'PPIACO',
    'pce_level':           'PCEPI',
    'oil_wti':             'DCOILWTICO',
}

def fetch_fred(series_id):
    r = requests.get(FRED_BASE, params={
        'series_id': series_id, 'observation_start': START,
        'observation_end': END, 'api_key': FRED_API_KEY,
        'file_type': 'json', 'frequency': 'm',
        'aggregation_method': 'eop', 'units': 'lin',
    })
    r.raise_for_status()
    obs = r.json()['observations']
    df = pd.DataFrame(obs)[['date','value']]
    df['date']  = pd.to_datetime(df['date']) + pd.offsets.MonthEnd(0)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    return df.set_index('date')['value']

fred_df = pd.DataFrame({name: fetch_fred(sid) for name, sid in FRED_SERIES.items()})
fred_df['cpi_yoy'] = fred_df['cpi_level'].pct_change(12) * 100
fred_df['pce_yoy'] = fred_df['pce_level'].pct_change(12) * 100

fred_df.to_csv('data/raw/fred_data.csv')
print(f"FRED data saved: {fred_df.shape[0]} rows x {fred_df.shape[1]} cols")
fred_df.tail(3)

## 2. Philadelphia Fed Survey of Professional Forecasters (SPF)

No API key needed. Downloads directly from the Philadelphia Fed website.
`CPI2` = one-quarter-ahead median CPI forecast, forward-filled to monthly.

In [ ]:
SPF_URL = ('https://www.philadelphiafed.org/-/media/frbp/assets/surveys-and-data/'
           'survey-of-professional-forecasters/data-files/files/median_cpi_level.xlsx')

def load_spf():
    r = requests.get(SPF_URL, timeout=30)
    r.raise_for_status()
    xl = pd.read_excel(io.BytesIO(r.content), sheet_name=0)
    xl = xl[['YEAR','QUARTER','CPI2']].dropna(subset=['CPI2']).copy()
    xl.columns = ['year','quarter','spf_cpi_forecast']
    def to_me(row):
        return pd.Timestamp(year=int(row.year), month=int(row.quarter)*3, day=1) + pd.offsets.MonthEnd(0)
    xl['date'] = xl.apply(to_me, axis=1)
    xl = xl[xl.date >= '2021-01-01'].set_index('date')[['spf_cpi_forecast']]
    idx = pd.date_range('2021-01-31', '2026-04-30', freq='ME')
    return xl.reindex(idx).ffill().rename_axis('date')

spf_df = load_spf()
spf_df.to_csv('data/raw/spf_data.csv')
print(f"SPF data saved: {spf_df.shape[0]} rows")
spf_df.tail(6)

## 3. Kalshi Prediction Market Data

Needs a free Kalshi account: https://kalshi.com/sign-up

Set `KALSHI_EMAIL` and `KALSHI_PASSWORD` below (or in `.env`).
If credentials are missing, a placeholder CSV (all NaN) is created automatically — the rest of the pipeline still works.

In [ ]:
# ── SET YOUR KALSHI CREDENTIALS HERE ────────────────────────────
KALSHI_EMAIL    = os.getenv('KALSHI_EMAIL',    'YOUR_KALSHI_EMAIL_HERE')
KALSHI_PASSWORD = os.getenv('KALSHI_PASSWORD', 'YOUR_KALSHI_PASSWORD_HERE')
# ────────────────────────────────────────────────────────────────

BASE_K = 'https://trading-api.kalshi.com/trade-api/v2'

def kalshi_implied_cpi_for_month(markets_for_month):
    tp = []
    for m in markets_for_month:
        title = m.get('title','') + ' ' + m.get('subtitle','')
        match = re.search(r'(\d+\.?\d*)\s*%', title)
        if not match:
            continue
        thr  = float(match.group(1))
        res  = m.get('result','')
        prob = 1.0 if res=='yes' else (0.0 if res=='no' else float(m.get('yes_bid', 0.5)))
        tp.append((thr, prob))
    if len(tp) < 2:
        return None
    tp.sort()
    thr_vals = [x[0] for x in tp]
    probs    = [x[1] for x in tp]
    exp = sum((probs[i]-probs[i+1])*(thr_vals[i]+thr_vals[i+1])/2 for i in range(len(thr_vals)-1))
    exp += (1-probs[0])*(thr_vals[0]-0.25) + probs[-1]*(thr_vals[-1]+0.25)
    return round(exp, 3)

if 'YOUR_KALSHI' in KALSHI_EMAIL:
    print("Kalshi credentials not set — saving placeholder (all NaN).")
    print("Add credentials above and re-run this cell to get real data.")
    dates = pd.date_range('2022-01-31','2026-04-30', freq='ME')
    kalshi_df = pd.DataFrame({'kalshi_implied_cpi': np.nan}, index=dates)
    kalshi_df.index.name = 'date'
else:
    r = requests.post(f'{BASE_K}/login', json={'email': KALSHI_EMAIL, 'password': KALSHI_PASSWORD})
    r.raise_for_status()
    kh = {'Authorization': f'Bearer {r.json()["token"]}'}

    series_r = requests.get(f'{BASE_K}/series', headers=kh, params={'limit':200}).json()
    cpi_ser  = [s for s in series_r.get('series',[])
                if 'CPI' in s.get('ticker','').upper() or 'inflation' in s.get('title','').lower()]
    sticker  = cpi_ser[0]['ticker']
    print(f"Using Kalshi series: {sticker}")

    markets, cursor = [], None
    while True:
        params = {'series_ticker': sticker, 'status': 'settled', 'limit': 200}
        if cursor: params['cursor'] = cursor
        resp = requests.get(f'{BASE_K}/markets', headers=kh, params=params)
        data = resp.json()
        markets.extend(data.get('markets',[]))
        cursor = data.get('cursor')
        if not cursor: break

    MONTH_RE = re.compile(r'(Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+(\d{4})', re.IGNORECASE)
    by_month = {}
    for m in markets:
        hit = MONTH_RE.search(m.get('title',''))
        if hit: by_month.setdefault(hit.group(0),[]).append(m)

    rows = []
    for ms, ml in sorted(by_month.items()):
        imp = kalshi_implied_cpi_for_month(ml)
        if imp is not None:
            rows.append({'date': pd.to_datetime(ms)+pd.offsets.MonthEnd(0), 'kalshi_implied_cpi': imp})

    kalshi_df = pd.DataFrame(rows).set_index('date').sort_index()
    kalshi_df = kalshi_df[kalshi_df.index >= '2022-01-31']
    print(f"Kalshi data: {len(kalshi_df)} rows")

kalshi_df.to_csv('data/raw/kalshi_cpi.csv')
kalshi_df.head()

## 4. GDELT News Sentiment

No API key needed. Makes one request per month (~52 requests, 0.5s sleep between each — takes ~3 minutes).

Averages article-level tone scores for articles mentioning 'inflation CPI' each month.
Tone scale: negative = more negative news coverage.

In [ ]:
def gdelt_tone(year, month):
    start = pd.Timestamp(year=year, month=month, day=1)
    end   = start + pd.offsets.MonthEnd(0)
    try:
        r = requests.get('https://api.gdeltproject.org/api/v2/artlist/artlist',
            params={'query':'inflation CPI','mode':'artlist','maxrecords':250,'format':'json',
                    'startdatetime':start.strftime('%Y%m%d000000'),
                    'enddatetime':end.strftime('%Y%m%d235959')}, timeout=30)
        r.raise_for_status()
        arts = r.json().get('articles',[])
        tones = [float(a['tone']) for a in arts if 'tone' in a]
        return np.mean(tones) if tones else np.nan
    except:
        return np.nan

dates = pd.date_range('2022-01-31','2026-04-30', freq='ME')
gdelt_rows = []
for d in dates:
    t = gdelt_tone(d.year, d.month)
    gdelt_rows.append({'date': d, 'gdelt_tone': t})
    time.sleep(0.5)
    print(f"  {d.strftime('%Y-%m')}: {t:.3f}" if not np.isnan(t) else f"  {d.strftime('%Y-%m')}: no data")

gdelt_s = pd.DataFrame(gdelt_rows).set_index('date')['gdelt_tone']
print(f"\nGDELT: {gdelt_s.notna().sum()}/{len(gdelt_s)} months with data")

## 5. Google Trends

No API key needed. Monthly US search volume index for 'inflation' (0–100).

In [ ]:
from pytrends.request import TrendReq

pt = TrendReq(hl='en-US', tz=0, timeout=(10,25))
pt.build_payload(['inflation'], cat=0, timeframe='2022-01-01 2026-04-01', geo='US')
time.sleep(2)
gt_raw = pt.interest_over_time()

if not gt_raw.empty:
    gt_raw.index = gt_raw.index + pd.offsets.MonthEnd(0)
    google_s = gt_raw['inflation'].rename('google_trends_inflation')
else:
    google_s = pd.Series(dtype=float, name='google_trends_inflation')

print(f"Google Trends: {len(google_s)} monthly observations")

sentiment_df = pd.concat([gdelt_s, google_s], axis=1)
sentiment_df.columns = ['gdelt_tone','google_trends_inflation']
sentiment_df = sentiment_df[sentiment_df.index >= '2022-01-31']
sentiment_df.to_csv('data/raw/sentiment_data.csv')
print(f"Sentiment data saved: {sentiment_df.shape[0]} rows")
sentiment_df.tail(3)

## 6. Merge Monthly Panel

Joins all sources on month-end date.
- **Outcome variable:** `cpi_yoy_next` = CPI YoY at month t+1 (i.e., what we are predicting)
- All predictors are at month t (no look-ahead bias)
- Kalshi column will be NaN until credentials are added

In [ ]:
def load_raw(path):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index = pd.to_datetime(df.index) + pd.offsets.MonthEnd(0)
    df.index.name = 'date'
    return df

fred_raw   = load_raw('data/raw/fred_data.csv')
spf_raw    = load_raw('data/raw/spf_data.csv')
kalshi_raw = load_raw('data/raw/kalshi_cpi.csv')
sent_raw   = load_raw('data/raw/sentiment_data.csv')

panel = fred_raw.join([spf_raw, kalshi_raw, sent_raw], how='left')
panel = panel.drop(columns=['cpi_level','pce_level'], errors='ignore')
panel['cpi_yoy_next'] = panel['cpi_yoy'].shift(-1)
panel = panel.loc['2022-01-31':'2026-03-31']
panel = panel.dropna(subset=['cpi_yoy_next'])

panel.to_csv('data/processed/monthly_panel.csv')
print(f"Panel saved: {panel.shape[0]} rows x {panel.shape[1]} cols")
print(f"Date range: {panel.index[0].date()} to {panel.index[-1].date()}")
print("\nMissing values per column:")
display(panel.isnull().sum().rename('NaN count').to_frame())

## Done

Panel saved to `data/processed/monthly_panel.csv`.

Open `02_analysis.ipynb` to run the statistical analysis and generate all plots.